# 04 - Evaluation

This notebook covers the evaluation plan: NeMo Evaluator matrix planning, durable completion collection, direct judge fallback, and token accounting.


In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'scripts').exists():
    for parent in Path.cwd().parents:
        if (parent / 'scripts').exists() and (parent / 'pyproject.toml').exists():
            REPO_ROOT = parent
            break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

OUT = REPO_ROOT / 'tutorial' / '_outputs' / 'eval-smoke'
OUT.mkdir(parents=True, exist_ok=True)
SUBMIT_EVALUATOR_JOBS = False
RUN_DIRECT_JUDGE = False
print(OUT)


The Evaluator matrix can be inspected locally because the job builders are pure functions.


In [ ]:
from scripts.eval.register_evaluator_entities import AdapterRow
from scripts.eval.run_evaluation_matrix import build_singleaxis_jobs, build_pairwise_jobs, build_49b_pairwise_jobs

adapters = [
    AdapterRow('lora-nim-llama-3.2-3b-r16', 'meta/llama-3.2-3b-instruct', 'cust-demo-1', 'nim_curated'),
    AdapterRow('lora-nim-llama-3.2-3b-r32', 'meta/llama-3.2-3b-instruct', 'cust-demo-2', 'nim_curated'),
    AdapterRow('lora-nemo-usvcs-llama-3.2-3b-r16', 'meta/llama-3.2-3b-instruct', 'cust-demo-3', 'nemo_usvcs_curated'),
]

wave_a = build_singleaxis_jobs(adapters, 'default/stage3-singleaxis-rubric')
wave_b = build_pairwise_jobs(adapters, 'default/stage3-pairwise-tournament')
wave_c = build_49b_pairwise_jobs(adapters, 'default/stage3-pairwise-tournament')

print('Wave A jobs:', len(wave_a))
print('Wave B jobs:', len(wave_b))
print('Wave C jobs:', len(wave_c))
print(json.dumps(wave_a[0], indent=2))


Durable completions are collected before scoring. This lets you retry judge prompts without regenerating target answers.


In [ ]:
completion_cmd = [
    sys.executable,
    'scripts/eval/collect_completions.py',
    '--dataset', '/mnt/nvme2/peft/datasets/v2/nim_curated/test_set_with_context.jsonl',
    '--model', 'default/lora-nim-llama-3.2-3b-r16',
    '--run-id', 'tutorial-smoke',
    '--limit', '5',
    '--max-tokens', '8192',
]
print(' '.join(completion_cmd))


This local token-accounting smoke test uses the pairwise summarizer over tiny synthetic artifacts. It demonstrates how target-generation and judge-scoring costs stay separate.


In [ ]:
from scripts.eval.run_direct_kimi_pairwise import summarize_pairwise

pairwise_path = OUT / 'pairwise.jsonl'
errors_path = OUT / 'errors.jsonl'

pairwise_rows = [
    {
        'winner': 'left',
        'agreement': 'agree',
        'target_token_counts': {
            'left': {'total_tokens_raw': 120},
            'right': {'total_tokens_raw': 135},
        },
        'outcomes': [
            {'judge': {'token_counts': {'prompt_tokens': 80, 'completion_tokens_raw': 20, 'total_tokens_raw': 100}}},
            {'judge': {'token_counts': {'prompt_tokens': 80, 'completion_tokens_raw': 18, 'total_tokens_raw': 98}}},
        ],
    }
]
pairwise_path.write_text(''.join(json.dumps(row) + '\n' for row in pairwise_rows))
errors_path.write_text('')

summary = summarize_pairwise(pairwise_path, errors_path)
print(json.dumps(summary, indent=2))


Direct judge command shapes. Keep `--limit` small for smoke tests, then remove it for the full matrix.


In [ ]:
singleaxis_cmd = [
    sys.executable,
    'scripts/eval/run_direct_kimi_singleaxis.py',
    '--responses', '/mnt/nvme2/peft/evals/completions/nim_curated/llama-3.2-3b/lora-nim/r16/tutorial-smoke/responses.jsonl',
    '--eval-run-id', 'tutorial-smoke',
    '--limit', '5',
]
pairwise_cmd = [
    sys.executable,
    'scripts/eval/run_direct_kimi_pairwise.py',
    '--pair', 'base', '/path/to/base/responses.jsonl', 'lora-r16', '/path/to/lora/responses.jsonl',
    '--eval-run-id', 'tutorial-smoke',
    '--limit', '5',
]
print('single-axis:', ' '.join(singleaxis_cmd))
print('pairwise:', ' '.join(pairwise_cmd))


Finish by rolling up token usage across eval artifacts with `scripts/eval/summarize_token_usage.py`.
